# Gradient Descent for Logistic Regression

**DSAI 601 — Week 7.** You implement it; we check it against `statsmodels`.

There is no closed form for $\hat{\boldsymbol\beta}$ in logistic regression, so we optimize numerically using gradient descent. Here, we'll just implement the *vanilla* version, though note that there are many modified versions that perform much better.

### The objective

From the lecture, the log-likelihood is a **sum**:

$$\ell(\boldsymbol\beta) = \sum_{i=1}^n\Big[y_i\ln\pi_i + (1-y_i)\ln(1-\pi_i)\Big], \qquad \pi_i = \frac{1}{1+e^{-\mathbf{x}_i^T\boldsymbol\beta}}$$

Maximizing $\ell$ is the same as minimizing $-\ell$, so we run gradient *descent* on the **negative log-likelihood** $-\ell(\boldsymbol\beta)$. 

### The gradient

We are minimizing $-\ell$:

$$\boxed{\;\nabla(-\ell)(\boldsymbol\beta) = -X^T(\mathbf y - \boldsymbol\pi)\;}$$

This is a $p$-vector. Component $j$ is $\sum_i (\pi_i - y_i)x_{ij}$ — note $x_{i0}=1$ for the intercept.

### The algorithm

Pick a learning rate $\alpha>0$. Start at $\boldsymbol\beta=\mathbf 0$ and repeat:

$\boldsymbol\eta = X\boldsymbol\beta$   
$\boldsymbol\pi = \tfrac{1}{1+e^{-\boldsymbol\eta}}$   
$\mathbf g = -X^T(\mathbf y - \boldsymbol\pi)$   
$\boldsymbol\beta_{new} \leftarrow \boldsymbol\beta - \alpha\mathbf g$    
Check size of $\mathbf g$. 

Stop when $\|\mathbf g\|$ is small or you hit a maximum iteration count.

*Note:* because $\mathbf g$ is a sum, its size grows with $n$ — so a good $\alpha$ for one dataset will be wrong for a dataset of a different size. Expect to retune it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

---
## 1. Implement it

Fill in the four `# TODO`s. Keep everything vectorized — no loops over observations.

In [ ]:
def sigmoid(eta):
    """Elementwise 1/(1+exp(-eta))."""
    # TODO
    raise NotImplementedError


def neg_log_lik(X, y, beta):
    """Negative log-likelihood -l(beta).  A SUM, matching lecture 3.1."""
    # TODO -- clip pi into [1e-12, 1-1e-12] before taking logs, or you will hit log(0)
    raise NotImplementedError


def gradient(X, y, beta):
    """grad of -l(beta) = X^T (pi - y).  Returns a p-vector."""
    # TODO
    raise NotImplementedError

In [ ]:
def grad_descent(X, y, alpha=0.0625, n_iter=20_000, tol=1e-8):
    """Return (beta, history_of_negloglik).

    Start at beta = 0. Each iteration: compute the gradient, step against it,
    record -l. Stop early if max|gradient| < tol.
    """
    beta = np.zeros(X.shape[1])
    history = []
    for t in range(n_iter):
        g = gradient(X, y, beta)
        # TODO: record the current loss, check the stopping rule, take the step
        raise NotImplementedError
    return beta, np.array(history)

---
## 2. Test data

Hours studied vs. pass/fail — the same example used in lecture, so you can compare against the
IRLS numbers.

In [ ]:
x = np.array([1., 2, 3, 4, 5, 6, 7, 8])
y = np.array([0., 0, 0, 1, 0, 1, 1, 1])
X = np.column_stack([np.ones_like(x), x])     # intercept column first

beta_gd, hist = grad_descent(X, y, alpha=0.0625, n_iter=20_000)
print(f"gradient descent : beta = [{beta_gd[0]:.6f}, {beta_gd[1]:.6f}]")
print(f"iterations run   : {len(hist)}")
print(f"final -loglik    : {hist[-1]:.6f}  

## 3. Check against `statsmodels`

In [ ]:
fit = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(f"statsmodels      : beta = [{fit.params[0]:.6f}, {fit.params[1]:.6f}]   ({fit.fit_history['iteration']} IRLS iterations)")
print(f"gradient descent : beta = [{beta_gd[0]:.6f}, {beta_gd[1]:.6f}]   ({len(hist)} iterations)")
print(f"max abs difference: {np.abs(beta_gd - fit.params).max():.2e}")
print(f"\nodds ratio per hour: exp({beta_gd[1]:.4f}) = {np.exp(beta_gd[1]):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist)
ax[0].set(xlabel="iteration", ylabel="$-\\ell(\\beta)$", title="negative log-likelihood")
ax[1].loglog(np.abs(hist - hist[-1]) + 1e-16)
ax[1].set(xlabel="iteration", ylabel="$-\\ell + \\ell^*$", title="distance to optimum (log-log)")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Q1.** How many iterations did gradient descent need? `statsmodels` (IRLS) took 6. 

---
## 4. The learning rate

$\alpha$ is yours to choose, and the method is not forgiving about it.

In [ ]:
# try a few different values of alpha and see how the convergence changes
for a in []: ## TODO: try a few different step sizes
    b, h = grad_descent(X, y, alpha=a, n_iter=2000)
    print(f"alpha = {a:<8} beta = [{b[0]:>9.4f}, {b[1]:>7.4f}]   -loglik = {h[-1]:.6f}")

print("\nalpha = 0.125, last 6 iterates:")
b = np.zeros(2)
for _ in range(2000):
    b = b - 0.125 * gradient(X, y, b)
for _ in range(6):
    b = b - 0.125 * gradient(X, y, b)
    print(f"   beta = [{b[0]:8.4f}, {b[1]:7.4f}]   -loglik = {neg_log_lik(X, y, b):.6f}")

**Q2.** Describe what happens at $\alpha = 0.125$. It is not divergence to infinity and it is
not convergence — what is it? Why does a larger $\alpha$ make it worse rather than better?

---
## 5. Feature scaling

Gradient descent steps in the same direction for every coordinate, so it struggles when
predictors are on very different scales. Newton's method does not have this problem.

Standardize $x$, refit, then map the coefficients back:

$$z = \frac{x-\bar x}{s_x} \qquad\Longrightarrow\qquad \beta_1 = \frac{\beta_1^{(z)}}{s_x}, \qquad \beta_0 = \beta_0^{(z)} - \beta_1^{(z)}\frac{\bar x}{s_x}$$

In [ ]:
# standardize x and run gradient descent on the standardized scale
xbar, sx = x.mean(), x.std(ddof=0)
z = (x - xbar) / sx
Xs = np.column_stack([np.ones_like(z), z])

beta_z, hist_z = grad_descent(Xs, y, alpha=0.0625, n_iter=2000)

# map the standardized coefficients back to the original scale
b1 = beta_z[1] / sx
b0 = beta_z[0] - beta_z[1] * xbar / sx

# compare to statsmodels
print(f"on standardized scale : beta = [{beta_z[0]:.6f}, {beta_z[1]:.6f}]   ({len(hist_z)} iterations)")
print(f"mapped back           : beta = [{b0:.6f}, {b1:.6f}]")
print(f"statsmodels           : beta = [{fit.params[0]:.6f}, {fit.params[1]:.6f}]")
print(f"max abs difference    : {max(abs(b0 - fit.params[0]), abs(b1 - fit.params[1])):.2e}")

**Q3.** Standardizing changed nothing about the model — the fitted probabilities are
identical. So what exactly did it speed up?

**Q4.** Fill in the comparison, using what you observed:

| | gradient descent | Newton–Raphson / IRLS |
|---|---|---|
| uses | gradient only | gradient **and** Hessian |
| cost per iteration | cheap, $O(np)$ | solve a $p\times p$ system |
| iterations needed here | ? | 6 |
| tuning required | ? | none |
| sensitive to feature scaling | ? | no |